In [143]:
import pandas as pd
import torch
from torch_geometric.data import Data
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import CGConv

# Load the merged and normalized data
df = pd.read_csv("../../dataset/final_normalized_graph.csv")

unique_nodes = pd.concat([df['source_number'], df['destination_number']]).unique()
node_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(unique_nodes))}

# Apply mapping
src = df['source_number'].map(node_mapping).values
dst = df['destination_number'].map(node_mapping).values

edge_index = torch.tensor([src, dst], dtype=torch.long)

# Notice we stripped out actual_time, actual_distance, and day_of_week
feature_cols = [
    'is_carting', 
    'is_ftl', 
    # 'day_of_week', <--- REMOVED
    'start_hour', 
    'osrm_time', 
    'osrm_distance'
]
edge_attr = torch.tensor(df[feature_cols].values, dtype=torch.float32)

# Create y (The target to predict)
# Shape: [E, 1]
target_col = ['real_actual_time_factor']
y = torch.tensor(df[target_col].values, dtype=torch.float32)

#  Build the PyTorch Geometric Data object
train_data = Data(edge_index=edge_index, edge_attr=edge_attr, y=y)

print(f"Graph created with {train_data.num_nodes} nodes and {train_data.num_edges} edges.")
print(f"Edge feature dimensions: {train_data.edge_attr.shape[1]}")

Graph created with 1562 nodes and 6605 edges.
Edge feature dimensions: 5


/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/torch_geometric/data/data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_attr', 'edge_index', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


In [144]:
class DelhiveryGNN(nn.Module):
    def __init__(self, num_nodes, embed_dim=16, num_edge_features=5):
        super(DelhiveryGNN, self).__init__()
        
        self.node_emb = nn.Embedding(num_nodes, embed_dim)
        
        # LAYER 1: Learns from immediate neighbors
        self.conv1 = CGConv(channels=embed_dim, dim=num_edge_features)
        
        # LAYER 2: Learns from neighbors of neighbors (2-hop context)
        self.conv2 = CGConv(channels=embed_dim, dim=num_edge_features)

        self.conv3 = CGConv(channels=embed_dim, dim=num_edge_features)

        
        # Prediction Head
        predictor_input_size = (embed_dim * 2) + num_edge_features
        self.predictor = nn.Sequential(
            nn.Linear(predictor_input_size, 64), # Widened from 32
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(32, 1) 
        )

    def forward(self, edge_index, edge_attr):
        num_nodes = self.node_emb.num_embeddings
        node_ids = torch.arange(num_nodes, device=edge_index.device)
        x = self.node_emb(node_ids)
        
        # Message Passing Phase 1
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        
        # Message Passing Phase 2 (New)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        
        x = self.conv3(x, edge_index, edge_attr)
        x = F.relu(x)

        source_nodes = x[edge_index[0]] 
        dest_nodes = x[edge_index[1]]   
        
        regression_input = torch.cat([source_nodes, dest_nodes, edge_attr], dim=1)
        predictions = self.predictor(regression_input)
        
        return predictions
    
    def get_contextual_embeddings(self, edge_index, edge_attr):
        num_nodes = self.node_emb.num_embeddings
        node_ids = torch.arange(num_nodes, device=edge_index.device)
        x = self.node_emb(node_ids)
        
        # Run through the 3 layers to get the network-aware context
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv3(x, edge_index, edge_attr)
        x = F.relu(x)
        
        return x

In [145]:
# 1. Initialize the Model
# train_data.num_nodes ensures we create exactly enough embeddings for all hubs

model = DelhiveryGNN(
    num_nodes=train_data.num_nodes, 
    embed_dim=16, 
    num_edge_features=train_data.edge_attr.shape[1]
)

# 2. Setup the Optimizer and Loss Function
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
criterion = nn.SmoothL1Loss()

# 3. Execute the Epochs
epochs = 1500
print("Starting Training...")

model.train() # Set model to training mode
for epoch in range(epochs):
    # Clear old gradients
    optimizer.zero_grad()
    
    # Forward Pass: Make predictions
    predictions = model(train_data.edge_index, train_data.edge_attr)
    
    # Calculate Error (MSE between predictions and real target)
    loss = criterion(predictions, train_data.y)
    
    # Backpropagation: Calculate the derivatives
    loss.backward()
    
    # Optimizer Step: Update the embeddings and weights
    optimizer.step()
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Loss: {loss.item():.4f}")
    # print(f"loss = {loss.item():.4f}  ::::: epochs = {epochs}, lr = {lr}")
# print("Training Complete!")

Starting Training...
Epoch 020/1500 | Loss: 1.8659
Epoch 040/1500 | Loss: 1.4247
Epoch 060/1500 | Loss: 1.2443
Epoch 080/1500 | Loss: 1.0824
Epoch 100/1500 | Loss: 0.9209
Epoch 120/1500 | Loss: 0.8438
Epoch 140/1500 | Loss: 0.7869
Epoch 160/1500 | Loss: 0.7433
Epoch 180/1500 | Loss: 0.7230
Epoch 200/1500 | Loss: 0.7109
Epoch 220/1500 | Loss: 0.6959
Epoch 240/1500 | Loss: 0.6831
Epoch 260/1500 | Loss: 0.6823
Epoch 280/1500 | Loss: 0.6734
Epoch 300/1500 | Loss: 0.6644
Epoch 320/1500 | Loss: 0.6658
Epoch 340/1500 | Loss: 0.6610
Epoch 360/1500 | Loss: 0.6589
Epoch 380/1500 | Loss: 0.6561
Epoch 400/1500 | Loss: 0.6529
Epoch 420/1500 | Loss: 0.6524
Epoch 440/1500 | Loss: 0.6509
Epoch 460/1500 | Loss: 0.6482
Epoch 480/1500 | Loss: 0.6476
Epoch 500/1500 | Loss: 0.6452
Epoch 520/1500 | Loss: 0.6452
Epoch 540/1500 | Loss: 0.6410
Epoch 560/1500 | Loss: 0.6392
Epoch 580/1500 | Loss: 0.6377
Epoch 600/1500 | Loss: 0.6358
Epoch 620/1500 | Loss: 0.6320
Epoch 640/1500 | Loss: 0.6277
Epoch 660/1500 | Lo

In [146]:
# train(2000, 0.02)
# train(3000, 0.03)
# train(4000, 0.05)
# train(2000, 0.03)
# train(500, 0.05)
# train(1000, 0.03)

loss, is too much. switching from nn.MSELoss to nn.SmoothL1Loss

In [147]:
# Save the model state dictionary (the weights and the learned embeddings)
torch.save(model.state_dict(), "../../final_model/delhivery_gnn_weights.pth")
print("Model weights saved to 'delhivery_gnn_weights.pth'")

Model weights saved to 'delhivery_gnn_weights.pth'


## Testing on test_data

In [148]:
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Data
import joblib

# Load the fitted scale that was used on the training data
scaler = joblib.load('../../dataset/fitted_scaler.pkl')

# Preprocessing the test data, same as train data to match input of our model
# ==========================================
test_df = pd.read_csv("../../dataset/test_data.csv")
test_df = test_df.drop(columns=['is_day', 'is_night', 'is_cutoff'], errors='ignore')

grouping_columns = ['source_number', 'destination_number', 'is_carting', 'is_ftl', 'start_hour']
aggregation_logic = {
    'osrm_time': 'median',
    'osrm_distance': 'median',
    'actual_distance': 'median',
    'actual_time': 'median',
    'real_actual_time_factor': 'median'
}
# Create the structural test graph
merged_test_df = test_df.groupby(grouping_columns).agg(aggregation_logic).reset_index()


# Apply scaling, and node mapping
# ==========================================
continuous_cols = ['osrm_time', 'osrm_distance', 'actual_distance', 'actual_time']
merged_test_df[continuous_cols] = scaler.transform(merged_test_df[continuous_cols])

# Round off
columns_to_round = continuous_cols + ['real_actual_time_factor']
merged_test_df[columns_to_round] = merged_test_df[columns_to_round].round(4)

merged_test_df['src_mapped'] = merged_test_df['source_number'].map(node_mapping)
merged_test_df['dst_mapped'] = merged_test_df['destination_number'].map(node_mapping)

# Drop any trips involving brand-new hubs that the training model has never seen
merged_test_df = merged_test_df.dropna(subset=['src_mapped', 'dst_mapped'])

# Save the clean test graph just to have it
merged_test_df.to_csv("../../dataset/final_normalized_test_graph.csv", index=False)
print("Test data normalization and mapping complete!")

# CONVERT TO PYTORCH TENSORS
# ==========================================
edge_index = torch.tensor([
    merged_test_df['src_mapped'].astype(int).values, 
    merged_test_df['dst_mapped'].astype(int).values
], dtype=torch.long)

feature_cols = ['is_carting', 'is_ftl', 'start_hour', 'osrm_time', 'osrm_distance']
edge_attr = torch.tensor(merged_test_df[feature_cols].values, dtype=torch.float32)

y = torch.tensor(merged_test_df['real_actual_time_factor'].values, dtype=torch.float32).view(-1, 1)

test_data = Data(edge_index=edge_index, edge_attr=edge_attr, y=y)
print(f"Test Graph created with {test_data.num_edges} edges.")

# EVALUATE THE MODEL
# ==========================================
model = DelhiveryGNN(num_nodes=len(node_mapping), embed_dim=16, num_edge_features=5)
model.load_state_dict(torch.load("../../final_model/delhivery_gnn_weights.pth"))

model.eval()
criterion = nn.SmoothL1Loss() 

with torch.no_grad():
    test_predictions = model(test_data.edge_index, test_data.edge_attr)
    test_loss = criterion(test_predictions, test_data.y)

print(f"Final Test Loss: {test_loss.item():.4f}")

Test data normalization and mapping complete!
Test Graph created with 3820 edges.
Final Test Loss: 0.6659


In [149]:
with open ("../../dataset/test_data.csv", "r", encoding="utf-8") as file:
    raw_tests = file.read();
raw_tests = raw_tests.split("\n");

In [ ]:
import torch
import numpy as np

# 1. Initialize and load model
model = DelhiveryGNN(num_nodes=len(node_mapping), embed_dim=16, num_edge_features=5)
model.load_state_dict(torch.load("../../final_model/delhivery_gnn_weights.pth"))
model.eval()

# 2. Precompute the contextual graph state ONCE
print("Precomputing graph network context...")
with torch.no_grad():
    # Pass the test graph structure to generate the aware embeddings
    global_context_embeddings = model.get_contextual_embeddings(test_data.edge_index, test_data.edge_attr)
print("Contextual embeddings ready!")

def test_single_route(source_hub_raw, dest_hub_raw, is_carting, is_ftl, day, hour, raw_osrm_time, raw_osrm_distance):
    if source_hub_raw not in node_mapping or dest_hub_raw not in node_mapping:
        return "ERROR: One of these hubs is brand new and wasn't in the training data!"
    
    # Get the internal integer IDs
    src_idx = node_mapping[source_hub_raw]
    dst_idx = node_mapping[dest_hub_raw]
    
    # 1. Look up the PRECOMPUTED contextual embeddings (O(1) time)
    # .unsqueeze(0) changes shape from [16] to [1, 16] to match the batch dimension
    x_src = global_context_embeddings[src_idx].unsqueeze(0)
    x_dst = global_context_embeddings[dst_idx].unsqueeze(0)
    
    # 2. Scale the continuous features
    dummy_input = np.array([[raw_osrm_time, raw_osrm_distance, 0.0, 0.0]])
    scaled_values = scaler.transform(dummy_input)[0]
    scaled_time = scaled_values[0]
    scaled_dist = scaled_values[1]
    
    # 3. Create the final 5-feature vector (excluding 'day')
    edge_attr = torch.tensor([[is_carting, is_ftl, hour, scaled_time, scaled_dist]], dtype=torch.float32)
    
    # 4. Run ONLY the predictor head
    with torch.no_grad():
        regression_input = torch.cat([x_src, x_dst, edge_attr], dim=1)
        predicted_factor = model.predictor(regression_input).item()
        
    # 5. Calculate the True ETA
    predicted_factor = min(predicted_factor, 4.0)
    true_eta = raw_osrm_time * predicted_factor
    
    return true_eta

# testing_str = """
# test,85,2,1,0,2,4,0,1,0,22.0,26.8778,21.570958256685618,66.0,3.0

# """
# [hub_A, hub_B, is_carting, is_ftl, day_of_week, start_hour, is_day, is_night, is_cutoff, osrm_time, osrm_distance,actual_distance,actual_time,real_actual_time_factor] = map(lambda x: float(x), (testing_str.strip()).split(',')[1:]);

# BARE MODEL TESTING FOR SELF SATISFRACTION
# ==========================================

# Pick two valid hubs from your dataset (replace 0 and 1 with actual hubs if needed)
# hub_A = 4
# hub_B = 39

# true_eta = float(test_single_route(
#     source_hub_raw=hub_A, 
#     dest_hub_raw=hub_B, 
#     is_carting=is_carting, is_ftl=is_ftl, 
#     day=day_of_week, hour=start_hour, 
#     raw_osrm_time=osrm_time, raw_osrm_distance=osrm_distance
# ))

# print(actual_time)
# print((true_eta<=actual_time*1.15) and (true_eta>=actual_time*0.85))

# Scenario 2: An FTL truck on Friday (Day 4) at 11:00 PM (Night traffic)
# print("\n[Scenario 2: FTL | Friday Night]")
# test_single_route(
#     source_hub_raw=hub_A, 
#     dest_hub_raw=hub_B, 
#     is_carting=0, is_ftl=1, 
#     day=4, hour=23, 
#     raw_osrm_time=12.5, raw_osrm_distance=400.0
# )

accuracy_count = 0
num_rows = 0;
for testing_str in raw_tests[1:]:  # Skip header row
    try:
        [hub_A, hub_B, is_carting, is_ftl, day_of_week, start_hour, is_day, is_night, is_cutoff, osrm_time, osrm_distance, actual_distance, actual_time, real_actual_time_factor] = map(lambda x: float(x), (testing_str.strip()).split(',')[1:])
        true_eta = float(test_single_route(
            source_hub_raw=hub_A, 
            dest_hub_raw=hub_B, 
            is_carting=is_carting, is_ftl=is_ftl, 
            day=day_of_week, hour=start_hour, 
            raw_osrm_time=osrm_time, raw_osrm_distance=osrm_distance
        ))
        
        # Check if prediction is within ±15% of actual time
        if ((true_eta <= actual_time * 1.15) and (true_eta >= actual_time * 0.85)):
            accuracy_count += 1
        num_rows+=1;
    except: pass

print(f"Accuracy: {accuracy_count}/{num_rows} predictions within ±15% tolerance")

Precomputing graph network context...
Contextual embeddings ready!


/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/v

Accuracy: 4501/7084 predictions within ±15% tolerance


/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/v

In [151]:
import joblib
joblib.dump(node_mapping, '../../dataset/node_mapping.pkl')

['../../dataset/node_mapping.pkl']